# 03 — Cleaning, splitting and train-only preprocessing

**Objective.** Apply the documented cleaning rules, create a stratified holdout and prove that
preprocessing statistics are learned from training rows only.

**Contract.** 80/20 stratified split (seed 42). The 20% holdout is used once for final
evaluation in notebook 05. `test.csv` is never used for selection.

In [1]:
import sys, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.cirrhosis import config, paths, data, features, preprocessing, models, evaluation, inference

paths.ensure_dirs()
for sub in ("data_quality", "cleaning", "experiments", "final_evaluation"):
    (paths.REPORTS_DIR / sub).mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "figure.autolayout": True})
SEED = config.RANDOM_SEED
print("Environment ready; seed =", SEED)

Environment ready; seed = 42


In [2]:
from sklearn.model_selection import train_test_split

train_raw, test_raw, submission = data.load_raw()
clean, audit_table = data.clean_training_data(train_raw)
display(audit_table)
audit_table.to_csv(paths.REPORTS_DIR / "cleaning" / "before_after_audit.csv", index=False)
print("Rows removed:", len(train_raw) - len(clean))
print("Out-of-range ages flagged (kept):", int(clean["age_imputed_flag"].sum()))
assert set(clean[config.TARGET].unique()) == set(config.CLASS_LABELS)

,check,before,after
0,Total records,15000.0,14999.0000
1,Missing values,62861.0,62861.0000
2,Invalid target labels,1.0,0.0000
3,Out-of-range ages,14.0,14.0000
4,Records retained (%),100.0,99.9933


Rows removed: 1
Out-of-range ages flagged (kept): 14


In [3]:
X = data.feature_matrix(clean)
y = clean[config.TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=config.TEST_SIZE, stratify=y, random_state=SEED
)
print("train split:", X_train.shape, "| validation split:", X_val.shape)
print("class balance train:", y_train.value_counts(normalize=True).round(4).to_dict())
print("class balance val:  ", y_val.value_counts(normalize=True).round(4).to_dict())
train_out = X_train.copy(); train_out[config.ID_COLUMN] = train_raw.loc[X_train.index, config.ID_COLUMN].to_numpy(); train_out[config.TARGET] = y_train
val_out = X_val.copy(); val_out[config.ID_COLUMN] = train_raw.loc[X_val.index, config.ID_COLUMN].to_numpy(); val_out[config.TARGET] = y_val
train_out.to_csv(paths.PROCESSED_DIR / "train.csv", index=False)
val_out.to_csv(paths.PROCESSED_DIR / "val.csv", index=False)

train split: (11999, 18) | validation split: (3000, 18)
class balance train: {'C': 0.6702, 'D': 0.3044, 'CL': 0.0254}
class balance val:   {'C': 0.6703, 'D': 0.3043, 'CL': 0.0253}


In [4]:
# Fit the full feature+preprocessing stack on TRAIN ONLY and verify its statistics.
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ("features", features.FeatureEngineeringTransformer(include_n_days=True)),
    ("preprocessor", preprocessing.build_preprocessor(include_n_days=True)),
])
Xt = pipe.fit_transform(X_train)
prep = pipe.named_steps["preprocessor"]
print("transformed shape:", Xt.shape)
print("finite:", np.isfinite(Xt.astype(float)).all())

num_imputer = prep.named_transformers_["num"].named_steps["impute"]
learned_medians = num_imputer.statistics_[: len(features.numeric_columns(True))]
expected = pipe.named_steps["features"].transform(X_train)[features.numeric_columns(True)].median().to_numpy()
print("max |learned median - train median|:", np.nanmax(np.abs(learned_medians - expected)))
assert np.allclose(learned_medians, expected, equal_nan=True)
print("PASS: preprocessing fitted on training rows only")

transformed shape: (11999, 46)
finite: True
max |learned median - train median|: 0.0
PASS: preprocessing fitted on training rows only


## Decisions & next steps

- One invalid label row was removed; extreme ages were flagged and clipped rather than deleted.
- The stratified holdout preserves the 67/30/2.5 class mix, so validation log loss is comparable
  to CV log loss.
- Verified: imputation medians equal training-set medians exactly, and no test rows participate.
- Next: benchmark nine classifiers with 5-fold stratified CV (notebook 04).